In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "telecom_guide.pdf"

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages from pdf")
print("\n ------First page preview (first 500 chars)-----")
print(pages[0].page_content[:500])

Loaded 9 pages from pdf

 ------First page preview (first 500 chars)-----
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(pages)

len(chunks)

37

In [5]:
chunks[0]

Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1')

In [9]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready {vector_store._collection.count()} vector stored")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store ready 37 vector stored


In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

test_query = "what is VOLTE? How does it improve call quality"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"----chunk {i} -----")
    print(doc.page_content[:300])
    print("\n")

----chunk 1 -----
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 


----chunk 2 -----
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt


----chunk 3 -----
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam




In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

# --- LLM via Groq API ---
llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash",temperature=0)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [17]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


A: Based on the provided context, here is how international roaming works and what charges you can expect:

### **How International Roaming Works**
* **Connection & Authentication:** When you travel outside your home network's coverage area, your device connects to a partner network in the visited country. The visited network authenticates your device using an inter-operator signaling protocol (SS7 or Diameter).
* **Validation:** Your home network validates your subscription and authorizes service.
* **Traffic Tunneling:** All voice, SMS, and data traffic is tunneled back to your home network for billing purposes, which introduces some latency compared to local network usage.

---

### **Charges to Expect**
Charges depend on the **Roaming Zone** of the country you are visiting:
* **Zone A (EU, UK, Australia, New Zealand):** Has the lowest roaming rates.
* **Zone B (USA, Canada, Japan, Singapore):** Has moderate rates.
* **Zone C (Rest of World):** Has the highest per-MB and per-minute 